# Facade spike: SDXL + ControlNet (Canny)
Colab 런타임을 **GPU(T4)** 로 바꾼 뒤 위에서부터 순서대로 실행하세요. 처음 한 번은 모델 다운로드로 5~10분 걸려요.
스케치 3장(`modern.png`, `villa_elev.png`, `facade_crop.png`)을 업로드하면, 스케치마다 재질 4종 = 총 12장을 만들어요.

In [ ]:
!pip -q install diffusers transformers accelerate safetensors opencv-python-headless

In [ ]:
import os, shutil
os.makedirs('sketches', exist_ok=True)
for n in ['modern.png', 'villa_elev.png', 'facade_crop.png']:   # Colab 왼쪽 파일 패널의 /content/sample_data 에 올려둔 파일
    shutil.copy(f'/content/sample_data/{n}', f'sketches/{n}')
print(os.listdir('sketches'))

In [ ]:
import torch, cv2, numpy as np
from PIL import Image
from diffusers import ControlNetModel, StableDiffusionXLControlNetPipeline, AutoencoderKL

controlnet = ControlNetModel.from_pretrained("diffusers/controlnet-canny-sdxl-1.0", torch_dtype=torch.float16,
    variant="fp16", use_safetensors=True)
vae = AutoencoderKL.from_pretrained("madebyollin/sdxl-vae-fp16-fix", torch_dtype=torch.float16)
pipe = StableDiffusionXLControlNetPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0", controlnet=controlnet, vae=vae,
    torch_dtype=torch.float16, variant="fp16", use_safetensors=True)
pipe.to("cuda")   # T4 VRAM 16GB에 들어감. CPU 오프로드는 시스템 RAM을 더 써서 제외

In [ ]:
# ---- 조절할 값 ----
CONTROL_SCALE = 0.8      # 높을수록 스케치에 충실, 낮을수록 AI가 자유롭게
CANNY_LOW, CANNY_HIGH = 50, 150   # 연필 질감이 너무 많이 잡히면 올리고, 선이 끊기면 내리세요
STEPS, SEED, MAXSIDE = 30, 42, 1024
EXTRA = ""               # 예: "at dusk, surrounded by forest"

MATERIALS = {
    "concrete": "exposed concrete finish, modern industrial look",
    "brick":    "red brick facade, warm classic look",
    "glass":    "full-height glass curtain wall, maximum openness",
    "wood":     "timber louvers and wood cladding, natural texture",
}
PROMPT = ("photorealistic architectural exterior photograph, {m}, natural daylight, "
          "professional architecture photography, highly detailed {extra}")
NEG = "sketch, drawing, line art, cartoon, painting, lowres, blurry, text, watermark, extra floors, distorted geometry"

def control_image(path):
    im = Image.open(path).convert("RGB")
    s = MAXSIDE / max(im.size)
    w, h = (int(im.width * s) // 8 * 8, int(im.height * s) // 8 * 8)
    im = im.resize((w, h), Image.LANCZOS)
    g = cv2.GaussianBlur(cv2.cvtColor(np.array(im), cv2.COLOR_RGB2GRAY), (3, 3), 0)
    e = cv2.Canny(g, CANNY_LOW, CANNY_HIGH)
    return Image.fromarray(np.stack([e] * 3, -1)), w, h

In [ ]:
import glob, zipfile
os.makedirs('out', exist_ok=True)
for path in sorted(glob.glob('sketches/*')):
    name = os.path.splitext(os.path.basename(path))[0]
    ctrl, w, h = control_image(path)
    ctrl.save(f'out/{name}__control.png')          # AI가 실제로 본 선화 (디버깅용)
    for key, m in MATERIALS.items():
        img = pipe(PROMPT.format(m=m, extra=EXTRA), negative_prompt=NEG, image=ctrl,
                   controlnet_conditioning_scale=CONTROL_SCALE, num_inference_steps=STEPS,
                   generator=torch.Generator("cpu").manual_seed(SEED)).images[0]
        img.save(f'out/{name}__{key}.png'); print('saved', name, key)

In [ ]:
# 결과 미리보기: 행 = 스케치, 열 = 원본 / 컨트롤 / 재질 4종
import matplotlib.pyplot as plt
names = sorted({os.path.basename(p).split('__')[0] for p in glob.glob('out/*__control.png')})
cols = ['control'] + list(MATERIALS)
fig, ax = plt.subplots(len(names), len(cols), figsize=(4 * len(cols), 4 * len(names)), squeeze=False)
for r, n in enumerate(names):
    for c, k in enumerate(cols):
        ax[r][c].imshow(Image.open(f'out/{n}__{k}.png')); ax[r][c].axis('off'); ax[r][c].set_title(f'{n} / {k}')
plt.tight_layout(); plt.show()

with zipfile.ZipFile('results.zip', 'w') as z:
    for f in glob.glob('out/*.png'): z.write(f)
files.download('results.zip')